In [2]:
# --- Imports ---

# Standard library modules used for retries, timing, URL handling, and local cache files.
import csv
import os
import re
import time
from urllib.parse import urljoin, urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import current_thread

# Browser automation stack.
from seleniumbase import Driver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [3]:
# --- Global Variables ---

# Target directory listing page on Clutch.
BASE_URL = "https://clutch.co"
url_param = "developers/shopify"
url = urljoin(BASE_URL, url_param)

# Parallelism settings: adjust these values to tune concurrency.
LINK_COLLECTION_WORKERS = 6
HTML_SAVE_WORKERS = 7

# Retry settings: higher values reduce skipped links but increase runtime.
PAGINATION_PAGE_RETRIES = 6
PAGINATION_LINK_RETRIES = 6
COMPANY_HTML_RETRIES = 6
INITIAL_RETRY_DELAY_SECONDS = 1.0
RETRY_BACKOFF_FACTOR = 2.0

# Resume-estimation settings.
ESTIMATED_LINKS_PER_PAGE = 50 # Average number of company links per pagination page.
RESUME_PROBE_PAGES = 12 # Number of pages to probe for estimating total links when resuming.
RESUME_SAFETY_PAGES = 5 # Number of pages to backtrack when resuming to ensure we don't miss any links.

# Cache and resume settings for saved HTML folders.
MAX_HTML_FOLDER_AGE_DAYS = 7
ETA_WINDOW_SIZE = 10

# Persistent cache for collected profile links.
LINKS_CACHE_CSV_PATH = "../data/bronze/company_profile_links_cache.csv"

# Base output directory for HTML snapshots.
BRONZE_HTML_BASE_DIR = "../data/bronze"

# --- SeleniumBase Initialization (UC Mode + Headless) ---
# UC mode helps reduce bot detection; headless keeps runs lightweight.
driver = Driver(
    uc=True,
    headless=True,
    incognito=True,
    chromium_arg="--blink-settings=imagesEnabled=false", # Turn off images
)
driver.get(url)

print("SeleniumBase UC driver started in headless mode.")

# This will be populated after reading pagination.
last_page_number = None

SeleniumBase UC driver started in headless mode.


### Functions

In [4]:
def extract_company_slug_from_profile_url(profile_url):
    """Extract the company slug from a Clutch profile URL."""

    parsed_url = urlparse(profile_url)
    path = parsed_url.path.rstrip("/")
    if "/profile/" in path:
        return path.split("/profile/")[-1]
    return path.split("/")[-1] or "page"


def extract_company_slug_from_html_filename(file_name):
    """Extract the company slug from a saved HTML file name."""

    match = re.match(r"^(?P<slug>.+?)_\d{8}_\d{6}_\d{3}\.html$", file_name)
    if match:
        return match.group("slug")
    return None


def format_eta(seconds):
    """Format ETA seconds into HH:MM:SS."""

    seconds = max(0, int(seconds))
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"


def calculate_eta_from_progress_samples(progress_samples):
    """Estimate ETA from the percentage change observed in the last samples.

    The function uses the first and last value in the current window, which
    makes the estimate stable and aligned with the user's request.
    """

    if len(progress_samples) < 2:
        return None

    first_time, first_percent = progress_samples[0]
    last_time, last_percent = progress_samples[-1]
    delta_time = last_time - first_time
    delta_percent = last_percent - first_percent

    if delta_time <= 0 or delta_percent <= 0:
        return None

    percent_per_second = delta_percent / delta_time
    if percent_per_second <= 0:
        return None

    remaining_percent = max(0.0, 100.0 - last_percent)
    return remaining_percent / percent_per_second


def load_recent_processed_company_slugs(bronze_dir, max_age_days=MAX_HTML_FOLDER_AGE_DAYS):
    """Load company slugs from recent HTML folders created in the bronze layer.

    Only folders named like `html_YYYYMMDD_HHMMSS` and modified within the age
    window are considered. This lets the scraper continue from previously saved
    HTML runs without re-collecting those companies from CSV.
    """

    if not os.path.exists(bronze_dir):
        return set()

    cutoff_seconds = max_age_days * 24 * 60 * 60
    now = time.time()
    processed_slugs = set()

    for entry_name in os.listdir(bronze_dir):
        entry_path = os.path.join(bronze_dir, entry_name)
        if not os.path.isdir(entry_path):
            continue
        if not entry_name.startswith("html_"):
            continue

        folder_age_seconds = now - os.path.getmtime(entry_path)
        if folder_age_seconds > cutoff_seconds:
            continue

        for file_name in os.listdir(entry_path):
            if not file_name.endswith(".html"):
                continue
            slug = extract_company_slug_from_html_filename(file_name)
            if slug:
                processed_slugs.add(slug)

    return processed_slugs


def load_cached_links(csv_path):
    """Load cached profile links from CSV if it exists.

    Returns a de-duplicated list preserving the original order from disk.
    """

    if not os.path.exists(csv_path):
        return []

    cached_links = []
    seen = set()

    with open(csv_path, "r", encoding="utf-8", newline="") as csv_file:
        reader = csv.DictReader(csv_file)
        for row in reader:
            link = (row.get("profile_url") or "").strip()
            if link and link not in seen:
                seen.add(link)
                cached_links.append(link)

    return cached_links


def append_links_to_cache(csv_path, new_links):
    """Append new profile links to the CSV cache without removing existing rows."""

    if not new_links:
        return 0

    file_exists = os.path.exists(csv_path)
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)

    appended_count = 0
    with open(csv_path, "a", encoding="utf-8", newline="") as csv_file:
        fieldnames = ["profile_url"]
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)

        if not file_exists:
            writer.writeheader()

        for link in new_links:
            cleaned_link = (link or "").strip()
            if not cleaned_link:
                continue

            writer.writerow({"profile_url": cleaned_link})
            appended_count += 1

    return appended_count

def estimate_resume_page(base_url, last_page_number, known_links):
    """Estimate the first page that may still contain unseen links.

    Hybrid logic:
    1. Probe early pages to find the first page that already contains cached links.
    2. Estimate a jump using: cached_rows / estimated_links_per_page.
    3. Jump forward, then step back a few safety pages.
    4. Validate forward until the first page with unseen links is found.
    5. If all validated pages are already cached, return last_page_number + 1.

    The CSV is append-only; this function only decides where to resume
    based on an in-memory list of links that are still worth scraping.
    """

    if not known_links:
        return 1

    print("Cache found. Estimating resume page with hybrid strategy...")

    known_links_set = set(known_links)

    def _fetch_page_links(page_number):
        """Fetch all profile links for one listing page."""
        probe_driver = Driver(
            uc=True,
            headless=True,
            incognito=True,
        )

        try:
            if page_number == 1:
                page_url = base_url
            else:
                separator = "&" if "?" in base_url else "?"
                page_url = f"{base_url}{separator}page={page_number}"

            probe_driver.get(page_url)
            WebDriverWait(probe_driver, 20).until(
                EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, "a.provider__title-link.directory_profile[href]")
                )
            )

            page_elements = probe_driver.find_elements(
                By.CSS_SELECTOR,
                "a.provider__title-link.directory_profile[href]",
            )
            return [
                element.get_attribute("href")
                for element in page_elements
                if element.get_attribute("href")
            ]
        finally:
            probe_driver.quit()

    # Step 1: find the first early page that intersects with cached links.
    first_cached_page = None
    probe_limit = min(last_page_number, RESUME_PROBE_PAGES)

    for page_number in range(1, probe_limit + 1):
        page_links = _fetch_page_links(page_number)
        if page_links and any(link in known_links_set for link in page_links):
            first_cached_page = page_number
            print(f"Found cached overlap on page {page_number}.")
            break

    if first_cached_page is None:
        print("No cached overlap found in early probe pages. Starting from page 1.")
        return 1

    # Step 2: estimate jump using cached rows and typical links per page.
    estimated_pages_from_cache = max(1, len(known_links) // max(1, ESTIMATED_LINKS_PER_PAGE))
    estimated_target_page = first_cached_page + estimated_pages_from_cache

    # Step 3: step back for safety to avoid missing boundary changes.
    candidate_start_page = max(1, estimated_target_page - RESUME_SAFETY_PAGES)
    candidate_start_page = min(candidate_start_page, last_page_number)

    print(
        f"Estimated target page: {estimated_target_page} "
        f"(cached_links={len(known_links)}, links_per_page={ESTIMATED_LINKS_PER_PAGE})."
    )
    print(f"Using safety window. Validation starts at page {candidate_start_page}.")

    # Step 4: validate forward and stop at first page with unseen links.
    for page_number in range(candidate_start_page, last_page_number + 1):
        page_links = _fetch_page_links(page_number)

        if not page_links:
            continue

        if any(link not in known_links_set for link in page_links):
            print(f"First page with unseen links estimated at: {page_number}")
            return page_number

    # Step 5: every validated page looked cached.
    return last_page_number + 1

In [5]:
def collect_links_from_page_with_worker(page_number, base_url):
    """Collect profile links from a single page using a dedicated worker driver.

    The worker retries the page collection with exponential backoff before
    returning a failure.
    """

    thread_name = current_thread().name

    # Build the page URL once; retries will target the same page.
    if page_number == 1:
        page_url = base_url
    else:
        separator = "&" if "?" in base_url else "?"
        page_url = f"{base_url}{separator}page={page_number}"

    for attempt in range(1, PAGINATION_LINK_RETRIES + 1):
        worker_driver = Driver(
            uc=True,
            headless=True,
            incognito=True,
        )

        attempt_start_time = time.time()

        try:
            print(
                f"[WORKER START] {thread_name} collecting links from page {page_number} "
                f"(attempt {attempt}/{PAGINATION_LINK_RETRIES})"
            )

            worker_driver.get(page_url)

            # Wait until profile link cards are visible.
            WebDriverWait(worker_driver, 20).until(
                EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, "a.provider__title-link.directory_profile[href]")
                )
            )

            page_elements = worker_driver.find_elements(
                By.CSS_SELECTOR, "a.provider__title-link.directory_profile[href]"
            )

            links = []
            for element in page_elements:
                href = element.get_attribute("href")
                if href:
                    links.append(href)

            elapsed_seconds = time.time() - attempt_start_time
            print(f"[WORKER DONE] {thread_name} collected page {page_number}")
            return {
                "page_number": page_number,
                "links": links,
                "error": None,
                "thread_name": thread_name,
                "duration_seconds": elapsed_seconds,
            }
        except Exception as exc:
            if attempt == PAGINATION_LINK_RETRIES:
                elapsed_seconds = time.time() - attempt_start_time
                print(
                    f"[WORKER ERROR] {thread_name} failed collecting page {page_number} "
                    f"after {PAGINATION_LINK_RETRIES} attempts: {exc}"
                )
                return {
                    "page_number": page_number,
                    "links": [],
                    "error": str(exc),
                    "thread_name": thread_name,
                    "duration_seconds": elapsed_seconds,
                }

            wait_seconds = INITIAL_RETRY_DELAY_SECONDS * (RETRY_BACKOFF_FACTOR ** (attempt - 1))
            print(
                f"[WORKER RETRY] {thread_name} page {page_number} failed on attempt {attempt}. "
                f"Waiting {wait_seconds:.1f}s..."
            )
            time.sleep(wait_seconds)
        finally:
            worker_driver.quit()

In [6]:
def capture_number_pages(driver, max_attempts=None):
    """Detect the last pagination page with retries and HTML fallback.

    Logic summary:
    1. Try to read page numbers from visible pagination elements.
    2. If elements are not fully rendered yet, parse `data-page` directly from page HTML.
    3. Retry with exponential backoff before failing.
    """

    if max_attempts is None:
        max_attempts = PAGINATION_PAGE_RETRIES

    # Backup parser for cases where data-page is present in HTML but not in elements yet.
    page_number_pattern = re.compile(r'data-page="(\d+)"')

    for attempt in range(1, max_attempts + 1):
        try:
            # Ensure DOM is available before reading pagination links.
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )

            # Scroll to trigger lazy-loaded pagination controls.
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(1.0)

            pagination_links = driver.find_elements(
                By.CSS_SELECTOR,
                'a.sg-pagination-v2-page-number[data-page]'
            )
            data_page_values = []

            # Read numeric page markers directly from elements.
            for link in pagination_links:
                value = link.get_attribute("data-page")
                if value and value.isdigit():
                    data_page_values.append(int(value))

            # Fallback: read data-page values directly from rendered HTML.
            if not data_page_values:
                html_matches = page_number_pattern.findall(driver.page_source)
                data_page_values = [int(match) for match in html_matches]

            if data_page_values:
                last_page_number = max(data_page_values)
                print(f"Pagination detected on attempt {attempt}. Last page: {last_page_number}")
                return last_page_number

            raise ValueError("No data-page values found in pagination")
        except Exception as exc:
            print(f"Pagination not ready on attempt {attempt}/{max_attempts}: {exc}")
            if attempt == max_attempts:
                raise

            # Exponential pause between attempts to avoid rapid retries.
            wait_seconds = INITIAL_RETRY_DELAY_SECONDS * (RETRY_BACKOFF_FACTOR ** (attempt - 1))
            time.sleep(wait_seconds)

In [6]:
def collect_company_profile_links(url, last_page_number, driver):
    """Collect company profile links from all pages using parallel execution.

    The CSV is treated as read-only here: links are never removed from it.
    We only skip links in memory when they were already covered by recent HTML
    folders, so the current run avoids scraping them again.
    """

    if last_page_number is None:
        raise ValueError("last_page_number must be defined before collecting company links.")

    cached_links = load_cached_links(LINKS_CACHE_CSV_PATH) or []
    processed_slugs = load_recent_processed_company_slugs(BRONZE_HTML_BASE_DIR) or set()

    # Build an in-memory worklist only. The CSV contents remain untouched.
    cached_links_to_scrape = []

    if processed_slugs:
        print(f"Found {len(processed_slugs)} recently processed company slugs in bronze HTML folders.")

    for link in cached_links:
        if extract_company_slug_from_profile_url(link) not in processed_slugs:
            cached_links_to_scrape.append(link)

    filtered_cached_links = cached_links_to_scrape

    if filtered_cached_links:
        print(f"Loaded {len(filtered_cached_links)} cached links from CSV after skipping recent HTML runs.")
    else:
        print("No usable cached links found. Starting from page 1.")

    # Estimate where unseen links may start if we already have cached data.
    start_page = estimate_resume_page(url, last_page_number, filtered_cached_links)

    if start_page > last_page_number:
        print("All pages appear to be fully cached. Reusing cached links.")
        return filtered_cached_links

    company_links = list(filtered_cached_links)
    seen_links = set(filtered_cached_links)
    total_pages_to_collect = last_page_number - start_page + 1
    completed = 0
    recent_progress_samples = []

    print(
        f"Starting link collection with {LINK_COLLECTION_WORKERS} workers "
        f"from page {start_page} to {last_page_number}..."
    )

    # Dispatch one worker task per page, starting at the estimated resume page.
    with ThreadPoolExecutor(max_workers=LINK_COLLECTION_WORKERS) as executor:
        future_map = {
            executor.submit(collect_links_from_page_with_worker, page_number, url): page_number
            for page_number in range(start_page, last_page_number + 1)
        }

        for future in as_completed(future_map):
            completed += 1
            result = future.result()
            thread_name = result.get("thread_name", "unknown-thread")
            page_number = result["page_number"]
            completion_percent = (completed / total_pages_to_collect) * 100 if total_pages_to_collect else 100
            recent_progress_samples.append((time.time(), completion_percent))
            if len(recent_progress_samples) > ETA_WINDOW_SIZE:
                recent_progress_samples.pop(0)
            eta_seconds = calculate_eta_from_progress_samples(recent_progress_samples)
            eta_text = format_eta(eta_seconds) if eta_seconds is not None else "calculating"

            if result["error"] is None:
                new_links = []
                # Deduplicate links while preserving insertion order.
                for href in result["links"]:
                    if href and href not in seen_links:
                        seen_links.add(href)
                        company_links.append(href)
                        new_links.append(href)

                # Incremental persistence: write new links immediately after each page.
                append_links_to_cache(LINKS_CACHE_CSV_PATH, new_links)

                print(
                    f"[PROGRESS] {completed}/{total_pages_to_collect} pages collected by {thread_name} "
                    f"(page {page_number}, new links: {len(new_links)}, "
                    f"completed: {completion_percent:.2f}%, ETA: {eta_text})"
                )
            else:
                print(
                    f"[PROGRESS] {completed}/{total_pages_to_collect} pages failed by {thread_name}: "
                    f"page {page_number} | {result['error']} "
                    f"(completed: {completion_percent:.2f}%, ETA: {eta_text})"
                )

    return company_links

In [7]:
def save_page_html_to_bronze(
    page_url,
    driver,
    bronze_dir="../data/bronze",
    max_retries=None,
    initial_delay=INITIAL_RETRY_DELAY_SECONDS,
    backoff_factor=RETRY_BACKOFF_FACTOR,
):
    """Save page HTML to bronze with automatic exponential backoff retry."""

    if max_retries is None:
        max_retries = COMPANY_HTML_RETRIES

    # Extract the profile slug from '/profile/<slug>' in the URL.
    parsed_url = urlparse(page_url)
    path = parsed_url.path.rstrip("/")
    if "/profile/" in path:
        profile_slug = path.split("/profile/")[-1]
    else:
        profile_slug = path.split("/")[-1] or "page"

    # Build a filesystem-safe file name with a high-resolution timestamp.
    safe_slug = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in profile_slug)

    os.makedirs(bronze_dir, exist_ok=True)

    for attempt in range(1, max_retries + 1):
        try:
            # Open the target page and wait until the DOM is available.
            driver.get(page_url)
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )

            timestamp = time.strftime("%Y%m%d_%H%M%S")
            millis = int((time.time() % 1) * 1000)
            file_name = f"{safe_slug}_{timestamp}_{millis:03d}.html"
            file_path = os.path.join(bronze_dir, file_name)

            with open(file_path, "w", encoding="utf-8") as html_file:
                html_file.write(driver.page_source)

            thread_name = current_thread().name
            return {
                "saved_file": file_path,
                "profile_slug": profile_slug,
                "thread_name": thread_name,
            }
        except Exception as exc:
            if attempt == max_retries:
                print(f"[ERROR] Failed after {max_retries} attempts: {page_url} | {exc}")
                raise

            # Wait longer after each failure.
            wait_seconds = initial_delay * (backoff_factor ** (attempt - 1))
            print(
                f"[RETRY] Attempt {attempt}/{max_retries} failed for {page_url}. "
                f"Waiting {wait_seconds:.1f}s..."
            )
            time.sleep(wait_seconds)

    raise RuntimeError(f"Failed to save HTML for {page_url}")

In [8]:
def create_headless_driver():
    """Create a dedicated SeleniumBase UC headless driver."""

    # A fresh driver per worker avoids shared-state race conditions.
    return Driver(
        uc=True,
        headless=True,
        incognito=True,
    )

In [8]:
def save_company_page_with_worker(company_url, bronze_dir="../data/bronze"):
    """Create a worker driver, save one company page, and close the driver."""

    worker_driver = create_headless_driver()
    thread_name = current_thread().name
    job_start_time = time.time()

    try:
        print(f"[WORKER START] {thread_name} handling {company_url}")
        saved_path = save_page_html_to_bronze(
            company_url,
            worker_driver,
            bronze_dir=bronze_dir,
            max_retries=COMPANY_HTML_RETRIES,
            initial_delay=INITIAL_RETRY_DELAY_SECONDS,
            backoff_factor=RETRY_BACKOFF_FACTOR,
        )
        elapsed_seconds = time.time() - job_start_time
        print(f"[WORKER DONE] {thread_name} completed {company_url}")
        return {
            "url": company_url,
            "saved_file": saved_path["saved_file"],
            "profile_slug": saved_path["profile_slug"],
            "error": None,
            "thread_name": thread_name,
            "duration_seconds": elapsed_seconds,
        }
    except Exception as exc:
        elapsed_seconds = time.time() - job_start_time
        print(f"[WORKER ERROR] {thread_name} failed {company_url}: {exc}")
        return {
            "url": company_url,
            "saved_file": None,
            "error": str(exc),
            "thread_name": thread_name,
            "duration_seconds": elapsed_seconds,
        }
    finally:
        # Always release browser resources.
        worker_driver.quit()

In [9]:
def save_all_company_pages_to_bronze(
    company_profile_links,
    bronze_dir=BRONZE_HTML_BASE_DIR,
):
    """Save all company profile HTML pages using parallel execution.

    At the start of each run, this function creates a dedicated output folder:
    `html_YYYYMMDD_HHMMSS` inside the bronze directory.
    """

    if not company_profile_links:
        raise ValueError("company_profile_links is empty. Run link collection first.")

    max_workers = HTML_SAVE_WORKERS

    # Create one run folder to keep HTML snapshots grouped by execution.
    run_timestamp = time.strftime("%Y%m%d_%H%M%S")
    run_html_dir = os.path.join(bronze_dir, f"html_{run_timestamp}")
    os.makedirs(run_html_dir, exist_ok=True)

    print(f"Starting HTML save with {max_workers} workers...")
    print(f"Run HTML directory: {run_html_dir}")

    saved_files = []
    failed_links = []
    total = len(company_profile_links)
    completed = 0
    recent_progress_samples = []

    # Dispatch one save task per company URL.
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_map = {
            executor.submit(
                save_company_page_with_worker,
                company_url,
                run_html_dir,
            ): company_url
            for company_url in company_profile_links
        }

        for future in as_completed(future_map):
            completed += 1
            result = future.result()
            thread_name = result.get("thread_name", "unknown-thread")
            completion_percent = (completed / total) * 100 if total else 100
            recent_progress_samples.append((time.time(), completion_percent))
            if len(recent_progress_samples) > ETA_WINDOW_SIZE:
                recent_progress_samples.pop(0)
            eta_seconds = calculate_eta_from_progress_samples(recent_progress_samples)
            eta_text = format_eta(eta_seconds) if eta_seconds is not None else "calculating"

            if result["error"] is None:
                saved_files.append(result["saved_file"])
                profile_slug = result.get("profile_slug", "unknown-company")
                print(
                    f"[PROGRESS] Saved company {profile_slug} by {thread_name} "
                    f"({completed}/{total}, completed: {completion_percent:.2f}%, ETA: {eta_text})"
                )
            else:
                failed_links.append({"url": result["url"], "error": result["error"]})
                print(
                    f"[PROGRESS] {completed}/{total} failed by {thread_name}: {result['url']} "
                    f"(completed: {completion_percent:.2f}%, ETA: {eta_text})"
                )

    summary = {
        "total_profiles": total,
        "saved_count": len(saved_files),
        "failed_count": len(failed_links),
        "saved_files": saved_files,
        "failed_links": failed_links,
        "run_html_dir": run_html_dir,
    }

    print("HTML save finished.")
    print(f"Saved: {summary['saved_count']} | Failed: {summary['failed_count']}")

    return summary

In [ ]:
# Run pipeline

# Step 1: detect how many paginated result pages exist.
last_page_number = capture_number_pages(driver)
print(f"Last page number: {last_page_number}")

# Step 2: collect all company profile URLs from listing pages.
company_profile_links = collect_company_profile_links(url, last_page_number, driver)
print(f"Collected links: {len(company_profile_links)}")

# Step 3: save each company profile HTML in a run-specific bronze folder.
save_summary = save_all_company_pages_to_bronze(
    company_profile_links,
    bronze_dir=BRONZE_HTML_BASE_DIR,
)

print(f"Run HTML directory: {save_summary['run_html_dir']}")
print(f"Saved files: {save_summary['saved_count']}")
print(f"Failed files: {save_summary['failed_count']}")

Pagination detected on attempt 1. Last page: 203
Last page number: 203
Found 137 recently processed company slugs in bronze HTML folders.
Loaded 9975 cached links from CSV after skipping recent HTML runs.
Cache found. Estimating resume page with hybrid strategy...
Found cached overlap on page 1.
Estimated target page: 200 (cached_links=9975, links_per_page=50).
Using safety window. Validation starts at page 195.
First page with unseen links estimated at: 195
Starting link collection with 6 workers from page 195 to 203...
[WORKER START] ThreadPoolExecutor-0_4 collecting links from page 199 (attempt 1/6)
[WORKER START] ThreadPoolExecutor-0_5 collecting links from page 200 (attempt 1/6)
[WORKER START] ThreadPoolExecutor-0_2 collecting links from page 197 (attempt 1/6)
[WORKER START] ThreadPoolExecutor-0_0 collecting links from page 195 (attempt 1/6)
[WORKER START] ThreadPoolExecutor-0_3 collecting links from page 198 (attempt 1/6)
[WORKER START] ThreadPoolExecutor-0_1 collecting links fro

Exception ignored in: <function Chrome.__del__ at 0x000001A5EB507E20>
Traceback (most recent call last):
  File "c:\Users\User\Desktop\Dados\Projetos\leadforge-directory-hunter\venv\Lib\site-packages\seleniumbase\undetected\__init__.py", line 637, in __del__
    self.quit()
  File "c:\Users\User\Desktop\Dados\Projetos\leadforge-directory-hunter\venv\Lib\site-packages\seleniumbase\undetected\__init__.py", line 586, in quit
    self.service.send_remote_shutdown_command()
  File "c:\Users\User\Desktop\Dados\Projetos\leadforge-directory-hunter\venv\Lib\site-packages\selenium\webdriver\common\service.py", line 140, in send_remote_shutdown_command
    request.urlopen(f"{self.service_url}/shutdown")
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\urllib\request.py", line 216, in urlopen
    return opener.open(url, data, timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.

[WORKER START] ThreadPoolExecutor-1_6 handling https://clutch.co/profile/viha-digital-commerce

^

^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\socket.py", line 843, in create_connection
    exceptions.clear()  # raise only the last error
    ^^^^^^^^^^^^^^^^^^
KeyboardInterrupt: 


HTML for solwey-consulting saved at 00:49:11 by ThreadPoolExecutor-1_0
[WORKER DONE] ThreadPoolExecutor-1_0 completed https://clutch.co/profile/solwey-consulting
[WORKER START] ThreadPoolExecutor-1_4 handling https://clutch.co/profile/agency-partner-interactive
